# 03 -- Collections: 50 Songs About a Keyword

Returns **50 globally most-played songs** matching a given keyword (love, war, happiness, loneliness, money),
using lyrics-based content filtering.

### Algorithm

Three approaches are compared:

1. **Baseline** -- exact keyword match: count occurrences of the keyword in each
   track's lyrics, apply a threshold, then sort by total play count.
2. **Word2Vec** -- expand the keyword with semantically similar tokens via a
   pre-trained word2vec model, then combine their lyric counts.
3. **Classification** -- label tracks as "about X" vs "not about X" using
   keyword presence, train a classifier on the labelled set, then predict
   scores for the remaining tracks.

### Output columns
| Column | Description |
|--------|-------------|
| `rank` | 1-50 index, by total play count (descending) |
| `artist` | Artist name |
| `title` | Track title |
| `play_count` | Total plays across all users |

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from src.data.loader import MySpotifyRecommender
from src.models.collections import (
    collection_baseline,
    collection_classification,
    collection_classification_compare,
    collection_word2vec,
)

In [2]:
rs = MySpotifyRecommender.from_files(Path.cwd().parent / "data", download=True, triplets_sample_rows=100_000)

Data dir    : /home/samy/MySpotify/data
Source      : cleaned CSVs (/home/samy/MySpotify/data/csv)

  tracks      (1000000, 4)
  genres      (280831, 3)
  triplets    (100000, 3)
  lyrics_long (16845822, 3)


---
## Research

In [3]:
KEYWORDS = ["love", "war", "happiness", "loneliness", "money"]

### Approach 1 -- Baseline (exact keyword match)

In [4]:
baseline_results = collection_baseline(rs, KEYWORDS, n=1, top_n=50)
for i in range(len(KEYWORDS)):
    keyword = KEYWORDS[i]
    result = baseline_results.get(keyword, "No results found")
    print(f"Keyword: {keyword}")
    display(result)
    print("\n")

Keyword: love


,artist,title,play_count
1,OneRepublic,Secrets,712.0
2,Five Iron Frenzy,Canada,552.0
3,Train,Marry Me,488.0
4,Tub Ring,Invalid,415.0
5,Train,Hey_ Soul Sister,403.0
6,Lil Wayne / Eminem,Drop The World,363.0
7,Florence + The Machine,Cosmic Love,339.0
8,Bill Withers,Make Love To Your Mind,322.0
9,Lady GaGa,Speechless,289.0
10,Panic At The Disco,Behind The Sea [Live In Chicago],269.0




Keyword: war


,artist,title,play_count
1,Taylor Swift,Love Story,238.0
2,Taylor Swift,You Belong With Me,143.0
3,Carpenters,(They Long To Be) Close To You,140.0
4,Miley Cyrus,Party In The U.S.A.,124.0
5,Jewel,Sometimes It Be That Way,116.0
6,Jack Johnson,Breakdown,102.0
7,Margot & The Nuclear So And So's,Dress Me Like a Clown,84.0
8,Stan Ridgway,Drive_ She Said,82.0
9,Five Finger Death Punch,Bad Company,78.0
10,Lily Allen,LDN,76.0




Keyword: happiness


'No results found'



Keyword: loneliness


'No results found'



Keyword: money


,artist,title,play_count
1,Fokofpolisiekar,Hemel Op Die Platteland,109.0
2,Yann Tiersen,A Quai,83.0
3,Céline Dion,Je ne vous oublie pas,78.0
4,Paris Combo,Prête A Porter,47.0
5,Macaco,Mama Tierra,29.0
6,Regina Spektor,Consequence Of Sounds,24.0
7,Belanova,Niño,22.0
8,Plastic Bertrand,Ca plane pour moi,21.0
9,Yelle,Amour Du Sol,21.0
10,Mecano,Sólo Soy Una Persona,20.0


### Approach 2 -- Word2Vec (expanded keywords)

In [5]:
w2v_results = collection_word2vec(rs, KEYWORDS, n=50, top_n=50)
for kw, df in w2v_results.items():
    print(f"\n{'='*50}")
    print(f"  Collection: {kw.upper()}")
    print(f"{'='*50}")
    display(df)


  Collection: LOVE


,artist,title,play_count
1,Travie McCoy,Billionaire [feat. Bruno Mars] (Explicit Albu...,266.0
2,Beastie Boys,Unite (2009 Digital Remaster),202.0
3,Eminem,Mockingbird,176.0
4,Guns N' Roses,Paradise City,170.0
5,Eminem,Without Me,166.0
6,Miley Cyrus,Party In The U.S.A.,124.0
7,Eminem / Nate Dogg,'Till I Collapse,121.0
8,Guns N' Roses,Don't Cry (Original),82.0
9,Aesop Rock,None Shall Pass (Main),72.0
10,Reality Check,Masquerade (Reality Check Album Version),63.0



  Collection: WAR


,artist,title,play_count



  Collection: HAPPINESS


,artist,title,play_count
1,The Roots,The Session (Longest Posse Cut In History_ 12:43),1.0
2,Jessica Simpson,I've Got My Eyes On You,0.0
3,Heather Small,I've Been There,0.0
4,Red Hot Chili Peppers,Around The World (Album Version),0.0
5,Jessica Simpson,I Think I'm In Love With You,0.0
6,New Radicals,Maybe You've Been Brainwashed Too,0.0
7,John Martyn,Could've Been Me,0.0
8,Spice Girls,Something Kinda Funny,0.0
9,Missing Persons,Give (Dance Mix) (2002 Digital Remaster),0.0
10,Jessica Simpson,My Wonderful,0.0



  Collection: LONELINESS


,artist,title,play_count
1,Silkk The Shocker,D-Game (feat. Master P_ Krazy and Terror) (Remix),0.0



  Collection: MONEY


,artist,title,play_count
1,The Hollies,I'm Down,5.0
2,Bruce Springsteen,I'm Goin' Down,2.0
3,Red Hot Chili Peppers,I Like Dirt (Album Version),1.0
4,The Benjamin Gate,Lay It Down (Untitled Album Version),0.0
5,Mad Skillz,It's Goin' Down [Explicit],0.0
6,Cupid,Cupid Shuffle [DFA Dub],0.0
7,Kool & The Gang,Jungle Boogie,0.0
8,Andres Calamaro,No Se Puede Vivir Del Amor,0.0
9,UGK (Underground Kingz),She Luv It,0.0
10,Unladylike,Sit Down,0.0


### Approach 3 -- Classification (MultinomialNB / Logistic / SGD / RandomForest on lyrics)

In [6]:
classifiers = ["nb", "logistic", "sgd", "forest"]
clf_results = collection_classification_compare(rs, KEYWORDS, n=10, neg_ratio=1, classifiers=classifiers)
for name in classifiers:
    print(f"\n{'='*60}\nClassifier: {name.upper()}\n{'='*60}")
    for kw, df in clf_results[name].items():
        print(f"\n  Collection: {kw.upper()}")
        display(df)


Classifier: NB

  Collection: LOVE


,artist,title,play_count
1,Blessid Union Of Souls,Could've Been With You,0.0
2,Maxwell,Arroz Con Pollo,0.0
3,Maxwell,Eachhoureachsecondeachminuteeachday:Of My Life,0.0
4,Joe Clay,Annabella,0.0
5,Maxwell,This Woman's Work,4.0
6,Whitesnake,Mistreated (Live) (2007 Digital Remaster),0.0
7,Ernest Tubb,Have You Ever Been Lonely (Have You Ever Been ...,0.0
8,Glenn Hughes,Mistreated,0.0
9,Hughes Turner Project,Mistreated,0.0
10,Songs:Ohia,Ghost Tropic,0.0



  Collection: WAR


,artist,title,play_count
1,All-4-One,Santa Claus Is Coming To Town (LP Version),0.0
2,Kenny G,Santa Claus Is Coming To Town,0.0
3,Matt Belsante,Santa Claus Is Coming To Town (White Christmas...,0.0
4,Willie Nelson,Here Comes Santa Claus,0.0
5,Gene Autry,Here Comes Santa Claus,0.0
6,Bob Dylan,Here Comes Santa Claus,0.0
7,Andrea Bocelli,Santa Claus Is Coming To Town,0.0
8,The SA,Santa Claus Is Coming to Town,0.0
9,Patty Loveless,The Boys Are Back In Town,0.0
10,Everclear,The Boys Are Back In Town,0.0



  Collection: HAPPINESS


...

,artist,title,play_count



  Collection: LONELINESS


,artist,title,play_count



  Collection: MONEY


,artist,title,play_count
1,Sniper,Panam All Star,0.0
2,Java,Danser,0.0
3,Mac Tyer,Outro,0.0
4,Alias,Hors la loi,0.0
5,Sniper,Y'a Pas De Mérite,0.0
6,La Fouine,Ma Tabatière (Chronique D'Un Dealer),0.0
7,Léo Ferré,Vitrines,0.0
8,Sniper,Fallait que je te dise (radio edit),0.0
9,Les Sages Poetes De La Rue,Teknik dans la peau,0.0
10,Youssoupha,Éternel recommencement,0.0


### Comparison

Check overlap between the three approaches for each keyword.

In [7]:
import pandas as pd

def summarize(df):
    if df is None or len(df) == 0:
        return 0, set()
    return len(df), set(zip(df["artist"], df["title"]))

methods = [("baseline", baseline_results), ("w2v", w2v_results)]
methods += [(f"clf_{name}", clf_results[name]) for name in classifiers]

rows = {}
for kw in KEYWORDS:
    rows[kw.upper()] = {label: summarize(res.get(kw))[0] for label, res in methods}

print("Result sizes (rows per keyword and method):")
display(pd.DataFrame(rows).T)

Result sizes (rows per keyword and method):


,baseline,w2v,clf_nb,clf_logistic,clf_sgd,clf_forest
LOVE,50,50,50,50,50,50
WAR,50,0,50,50,50,50
HAPPINESS,0,11,0,0,0,0
LONELINESS,0,1,0,0,0,0
MONEY,50,33,50,50,50,50
